# Neural Networks

## Learning Objectives
1. Implement a 2-layer MLP with full backpropagation from scratch in NumPy on XOR.
2. Build a production MLP in PyTorch with BatchNorm, Dropout, and OOM handling.
3. Apply the MLP to tabular classification with a full training pipeline.
4. Compare weight initialisation strategies and their effect on gradient flow.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1: 2-Layer MLP Backprop from Scratch (NumPy / XOR)


In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
def sigmoid_deriv(a): return a * (1 - a)
def bce(y, yhat):
    return -np.mean(y * np.log(yhat + 1e-8) + (1 - y) * np.log(1 - yhat + 1e-8))


class TwoLayerMLP:
    """Hand-coded 2-layer MLP with sigmoid activations and binary CE loss."""

    def __init__(self, in_dim: int, hidden: int, out_dim: int = 1, lr: float = 0.1):
        self.lr = lr
        self.W1 = np.random.randn(in_dim, hidden) * np.sqrt(2 / in_dim)
        self.b1 = np.zeros(hidden)
        self.W2 = np.random.randn(hidden, out_dim) * np.sqrt(2 / hidden)
        self.b2 = np.zeros(out_dim)

    def forward(self, X):
        self.X = X
        self.z1 = X @ self.W1 + self.b1
        self.a1 = sigmoid(self.z1)
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = sigmoid(self.z2)
        return self.a2

    def backward(self, y):
        n = len(y)
        delta2 = (self.a2 - y.reshape(-1, 1)) / n
        dW2 = self.a1.T @ delta2
        db2 = delta2.sum(axis=0)
        delta1 = (delta2 @ self.W2.T) * sigmoid_deriv(self.a1)
        dW1 = self.X.T @ delta1
        db1 = delta1.sum(axis=0)
        self.W1 -= self.lr * dW1; self.b1 -= self.lr * db1
        self.W2 -= self.lr * dW2; self.b2 -= self.lr * db2

    def train_step(self, X, y):
        yhat = self.forward(X)
        self.backward(y)
        return bce(y, yhat.ravel())


X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=np.float32)
y_xor = np.array([0, 1, 1, 0], dtype=np.float32)

mlp = TwoLayerMLP(in_dim=2, hidden=4, lr=1.0)
for step in range(5000):
    loss = mlp.train_step(X_xor, y_xor)
    if step % 1000 == 0:
        preds = (mlp.forward(X_xor).ravel() > 0.5).astype(int)
        acc = (preds == y_xor).mean()
        print(f"Step {step:4d} | loss: {loss:.4f} | acc: {acc:.2f}")

preds_final = (mlp.forward(X_xor).ravel() > 0.5).astype(int)
print(f"\nXOR predictions: {preds_final} (expected [0,1,1,0])")
print(f"All correct: {np.all(preds_final == y_xor)}")


## Level 2: nn.Module MLP with BatchNorm + Dropout (PyTorch + OOM)


In [ ]:
class ProductionMLP(nn.Module):
    """MLP with BatchNorm, Dropout, and configurable depth."""

    def __init__(self, in_dim: int, hidden_dims: list, out_dim: int,
                 dropout: float = 0.3):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                       nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


X_cls, y_cls = make_classification(n_samples=3000, n_features=30,
                                    n_informative=12, n_redundant=6, random_state=42)
scaler_nn = StandardScaler().fit(X_cls[:2400])
X_cls = scaler_nn.transform(X_cls)
X_tr = torch.FloatTensor(X_cls[:2400]); y_tr = torch.LongTensor(y_cls[:2400])
X_te = torch.FloatTensor(X_cls[2400:]).to(device)
y_te = torch.LongTensor(y_cls[2400:]).to(device)
cls_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=128, shuffle=True)

model_prod = ProductionMLP(30, [128, 64, 32], 2, dropout=0.3).to(device)
opt_prod = torch.optim.Adam(model_prod.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_prod = torch.optim.lr_scheduler.StepLR(opt_prod, step_size=10, gamma=0.5)
crit_prod = nn.CrossEntropyLoss()

for epoch in range(40):
    model_prod.train()
    for xb, yb in cls_loader:
        opt_prod.zero_grad()
        try:
            loss = crit_prod(model_prod(xb.to(device)), yb.to(device))
        except RuntimeError as exc:
            if "out of memory" in str(exc).lower():
                print("OOM: reduce batch_size"); torch.cuda.empty_cache(); continue
            raise
        loss.backward(); opt_prod.step()
    scheduler_prod.step()
    if epoch % 10 == 0:
        model_prod.eval()
        with torch.no_grad():
            acc = (model_prod(X_te).argmax(1) == y_te).float().mean().item()
        print(f"Epoch {epoch:2d} | val acc: {acc:.4f} | "
              f"lr: {opt_prod.param_groups[0]['lr']:.5f}")


## Real-World Example 1: Tabular Classification Pipeline


In [ ]:
from sklearn.datasets import load_breast_cancer

bc = load_breast_cancer()
X_bc = StandardScaler().fit_transform(bc.data)
y_bc = bc.target

X_bc_t = torch.FloatTensor(X_bc[:455]); y_bc_t = torch.LongTensor(y_bc[:455])
X_bc_v = torch.FloatTensor(X_bc[455:]).to(device)
y_bc_v = torch.LongTensor(y_bc[455:]).to(device)
bc_loader = DataLoader(TensorDataset(X_bc_t, y_bc_t), batch_size=32, shuffle=True)

model_bc = ProductionMLP(30, [64, 32], 2, dropout=0.2).to(device)
opt_bc = torch.optim.Adam(model_bc.parameters(), lr=5e-4)
crit_bc = nn.CrossEntropyLoss()

best_val_acc = 0.0
patience_counter = 0
PATIENCE = 10
for epoch in range(100):
    model_bc.train()
    for xb, yb in bc_loader:
        opt_bc.zero_grad()
        crit_bc(model_bc(xb.to(device)), yb.to(device)).backward()
        opt_bc.step()
    model_bc.eval()
    with torch.no_grad():
        val_acc = (model_bc(X_bc_v).argmax(1) == y_bc_v).float().mean().item()
    if val_acc > best_val_acc:
        best_val_acc = val_acc; patience_counter = 0
    else:
        patience_counter += 1
    if patience_counter >= PATIENCE:
        print(f"Early stop at epoch {epoch+1}"); break
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | val acc: {val_acc:.4f} | best: {best_val_acc:.4f}")

print(f"Final best validation accuracy: {best_val_acc:.4f}")


## Real-World Example 2: Architecture Grid Search


In [ ]:
architectures = {
    "Shallow [128]":     [128],
    "Medium [128,64]":   [128, 64],
    "Deep [128,64,32]":  [128, 64, 32],
    "Wide [256,256]":    [256, 256],
    "Narrow [32,32,32]": [32, 32, 32],
}

arch_results = {}
for arch_name, hidden_dims in architectures.items():
    m = ProductionMLP(30, hidden_dims, 2, dropout=0.2).to(device)
    o = torch.optim.Adam(m.parameters(), lr=5e-4)
    c = nn.CrossEntropyLoss()
    for epoch in range(50):
        m.train()
        for xb, yb in bc_loader:
            o.zero_grad()
            c(m(xb.to(device)), yb.to(device)).backward()
            o.step()
    m.eval()
    with torch.no_grad():
        val_acc = (m(X_bc_v).argmax(1) == y_bc_v).float().mean().item()
    n_params = sum(p.numel() for p in m.parameters())
    arch_results[arch_name] = (val_acc, n_params)
    print(f"{arch_name:>22}: acc={val_acc:.4f}, params={n_params:,}")

best_arch = max(arch_results, key=lambda k: arch_results[k][0])
print(f"\nBest architecture: {best_arch} (acc={arch_results[best_arch][0]:.4f})")


## Real-World Example 3: Weight Initialisation Comparison and Gradient Flow


In [ ]:
def make_mlp_with_init(init_name: str, in_dim=30, hidden=64, out_dim=2):
    model_i = nn.Sequential(
        nn.Linear(in_dim, hidden), nn.ReLU(),
        nn.Linear(hidden, hidden), nn.ReLU(),
        nn.Linear(hidden, out_dim),
    ).to(device)
    for layer in model_i:
        if isinstance(layer, nn.Linear):
            if init_name == "zeros":
                nn.init.zeros_(layer.weight)
            elif init_name == "random_normal":
                nn.init.normal_(layer.weight, std=1.0)
            elif init_name == "xavier":
                nn.init.xavier_uniform_(layer.weight)
            elif init_name == "he":
                nn.init.kaiming_uniform_(layer.weight, nonlinearity='relu')
    return model_i


init_names = ["zeros", "random_normal", "xavier", "he"]
grad_norm_after_first_batch = {}

xb_init, yb_init = next(iter(bc_loader))
xb_init, yb_init = xb_init.to(device), yb_init.to(device)
crit_init = nn.CrossEntropyLoss()

for init_name in init_names:
    m_i = make_mlp_with_init(init_name)
    m_i.train()
    loss_i = crit_init(m_i(xb_init), yb_init)
    loss_i.backward()
    total_norm = sum(
        p.grad.detach().norm().item() ** 2
        for p in m_i.parameters() if p.grad is not None
    ) ** 0.5
    grad_norm_after_first_batch[init_name] = total_norm
    print(f"Init={init_name:>14}: gradient norm = {total_norm:.4f}")

print("\nKey insight: zeros init -> zero gradients (symmetry breaking fails)")
print("He init works best with ReLU activations")
